# Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
2. Extract specific portion of the string from refund_reason using regexp_extract function
3. Extract date and time from the refund_timestamp
4. Write transformed data to the Silver schema in hive metastore

In [0]:
df = spark.table('asql_gizmobox_db_conn_catalog_ui.dbo.refunds')

display(df)

## 1. Extract specific portion of the string from refund_reason using split function

[Documentation split](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html#pyspark.sql.functions.split)

In [0]:
from pyspark.sql import functions as F

df_refund_reason = (
    df
    .select(
        'refund_id',
        'payment_id',
        'refund_timestamp',
        'refund_amount',
        F.split('refund_reason',':')[0].alias('refund_reason'),
        F.split('refund_reason',':')[1].alias('refund_source')
    )
)

display(df_refund_reason)

## 2. Extract specific portion of the string from refund_reason using regexp_extract function

[Documentation regexp_extract](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html#pyspark.sql.functions.regexp_extract)

In [0]:
df_refund_reason = (
    df
    .select(
        'refund_id',
        'payment_id',
        'refund_timestamp',
        'refund_amount',
        F.regexp_extract('refund_reason', '^([^:]+):', 1).alias('refund_reason'),
        F.regexp_extract('refund_reason', ':([^:]+)$', 1).alias('refund_source')
    )
)

display(df_refund_reason)

## 3. Extract date and time from the refund_timestamp

[to date](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html#pyspark.sql.functions.to_date)

In [0]:
df_refund_reason = (
    df
    .select(
        'refund_id',
        'payment_id',
        F.to_date('refund_timestamp').alias('refund_date'),
        F.date_format('refund_timestamp', 'HH:mm:ss').alias('refund_time'),
        'refund_amount',
        F.regexp_extract('refund_reason', '^([^:]+):', 1).alias('refund_reason'),
        F.regexp_extract('refund_reason', ':([^:]+)$', 1).alias('refund_source')
    )
)

display(df_refund_reason)

## 4. Write transformed data to the Silver schema in other database

In [0]:
df_refund_reason.writeTo('gizmobox.silver.py_refunds').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_refunds;

In [0]:
%sql
SELECT * FROM gizmobox.silver.refunds;

In [0]:
%sql
DESC EXTENDED gizmobox.silver.py_refunds;